In [1]:

# Import pandas for reading and analysing tabular data
import pandas as pd

# Import Path for handling file and folder locations
from pathlib import Path

# Define the expected location of our TCS CSV file
csv_path = Path("data/raw/TCS_NS.csv")

# Check the parent directory as well if the notebook runs from notebooks/
if not csv_path.exists():
    csv_path = Path("../data/raw/TCS_NS.csv")

# Stop execution with a clear message if the CSV cannot be found
if not csv_path.exists():
    raise FileNotFoundError("TCS_NS.csv was not found. Check the data/raw folder.")

# Read the CSV file into a Pandas DataFrame
# parse_dates converts the Date column into actual datetime values
tcs_data = pd.read_csv(csv_path, parse_dates=["Date"])

# Display a confirmation message
print("TCS dataset loaded successfully!")

# Display the number of rows and columns
print("Dataset shape:", tcs_data.shape)

# Display all column names
print("\nColumn names:")
print(tcs_data.columns.tolist())

# Display the first five records
print("\nFirst five records:")
display(tcs_data.head())


TCS dataset loaded successfully!
Dataset shape: (1236, 9)

Column names:
['Date', 'Adj Close', 'Close', 'Dividends', 'High', 'Low', 'Open', 'Stock Splits', 'Volume']

First five records:


,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-11-04,3037.658691,3508.649902,0.0,3513.250000,3505.000000,3512.949951,0.0,328991
1,2021-11-08,3032.551025,3502.750000,0.0,3548.000000,3481.199951,3548.000000,0.0,2526082
2,2021-11-09,3035.104980,3505.699951,0.0,3535.000000,3496.000000,3508.000000,0.0,1782692
3,2021-11-10,3016.577393,3484.300049,0.0,3519.699951,3480.199951,3498.000000,0.0,1516489
4,2021-11-11,3019.954590,3488.199951,0.0,3510.050049,3436.000000,3480.000000,0.0,1284514


In [2]:

# ============================================================
# TCS DATA QUALITY CHECKS
# Purpose: Inspect the downloaded stock dataset
# ============================================================

# Define the columns that represent stock prices
price_columns = ["Open", "High", "Low", "Close", "Adj Close"]

# Convert price columns to numeric values for safe calculations
# This creates a separate copy and does not modify tcs_data
price_data = tcs_data[price_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# ------------------------------------------------------------
# 1. CHECK MISSING VALUES
# ------------------------------------------------------------

# Count missing values in every column
missing_values = tcs_data.isnull().sum()

# Display the missing value count
print("MISSING VALUES")
display(missing_values.to_frame(name="Missing Count"))

# ------------------------------------------------------------
# 2. CHECK DUPLICATE RECORDS
# ------------------------------------------------------------

# Count rows having duplicate dates
duplicate_dates = tcs_data["Date"].duplicated().sum()

# Count completely duplicated rows
duplicate_rows = tcs_data.duplicated().sum()

# Display duplicate counts
print("\nDUPLICATE RECORDS")
print("Duplicate dates:", duplicate_dates)
print("Duplicate rows:", duplicate_rows)

# ------------------------------------------------------------
# 3. CHECK INVALID STOCK PRICES
# ------------------------------------------------------------

# Count prices that are zero or negative
invalid_prices = (price_data <= 0).sum()

# Display invalid price counts
print("\nINVALID PRICES")
display(invalid_prices.to_frame(name="Invalid Count"))

# ------------------------------------------------------------
# 4. CHECK OHLC PRICE CONSISTENCY
# ------------------------------------------------------------

# Check whether the daily High is lower than Open or Close
high_invalid = (
    (price_data["High"] < price_data["Open"]) |
    (price_data["High"] < price_data["Close"])
)

# Check whether the daily Low is higher than Open or Close
low_invalid = (
    (price_data["Low"] > price_data["Open"]) |
    (price_data["Low"] > price_data["Close"])
)

# Check whether the daily High is lower than the daily Low
range_invalid = price_data["High"] < price_data["Low"]

# Count records violating OHLC consistency
invalid_ohlc = (high_invalid | low_invalid | range_invalid).sum()

# Display the result
print("\nOHLC CONSISTENCY")
print("Invalid OHLC records:", invalid_ohlc)

# ------------------------------------------------------------
# 5. CHECK TRADING VOLUME
# ------------------------------------------------------------

# Convert Volume to numeric values
volume_data = pd.to_numeric(
    tcs_data["Volume"],
    errors="coerce"
)

# Count negative trading volume records
negative_volume = (volume_data < 0).sum()

# Display the result
print("\nTRADING VOLUME")
print("Negative volume records:", negative_volume)

# ------------------------------------------------------------
# 6. CHECK DATE ORDER
# ------------------------------------------------------------

# Check whether dates are arranged from oldest to newest
dates_sorted = tcs_data["Date"].is_monotonic_increasing

# Display the result
print("\nDATE ORDER")
print("Dates sorted chronologically:", dates_sorted)


MISSING VALUES


,Missing Count
Date,0
Adj Close,1
Close,1
Dividends,0
High,0
Low,0
Open,0
Stock Splits,0
Volume,0



DUPLICATE RECORDS
Duplicate dates: 0
Duplicate rows: 0

INVALID PRICES


,Invalid Count
Open,0
High,0
Low,0
Close,0
Adj Close,0



OHLC CONSISTENCY
Invalid OHLC records: 0

TRADING VOLUME
Negative volume records: 0

DATE ORDER
Dates sorted chronologically: True


In [3]:
# Display missing values in every column
print("MISSING VALUES BY COLUMN")
display(tcs_data.isnull().sum().to_frame("Missing Count"))

# Calculate the total number of missing cells
print("Total missing values:", tcs_data.isnull().sum().sum())

# Count duplicate dates
print("\nDuplicate dates:", tcs_data["Date"].duplicated().sum())

# Count completely duplicated rows
print("Duplicate rows:", tcs_data.duplicated().sum())

# Check zero-volume records
volume = pd.to_numeric(tcs_data["Volume"], errors="coerce")
print("Zero-volume records:", (volume == 0).sum())

MISSING VALUES BY COLUMN


,Missing Count
Date,0
Adj Close,1
Close,1
Dividends,0
High,0
Low,0
Open,0
Stock Splits,0
Volume,0


Total missing values: 2

Duplicate dates: 0
Duplicate rows: 0
Zero-volume records: 2


In [4]:

# ============================================================
# IDENTIFY PROBLEMATIC TCS RECORDS
# ============================================================

# Find all rows containing at least one missing value
missing_rows = tcs_data.loc[
    tcs_data.isnull().any(axis=1)
]

# Convert trading volume into numeric values
volume_numeric = pd.to_numeric(
    tcs_data["Volume"],
    errors="coerce"
)

# Find rows where trading volume equals zero
zero_volume_rows = tcs_data.loc[
    volume_numeric == 0
]

# Display records containing missing values
print("RECORDS WITH MISSING VALUES")
display(missing_rows)

# Display records with zero trading volume
print("\nRECORDS WITH ZERO VOLUME")
display(zero_volume_rows)

# Display the dates of all problematic records
print("\nDATES WITH MISSING VALUES:")
print(missing_rows["Date"].tolist())

print("\nDATES WITH ZERO VOLUME:")
print(zero_volume_rows["Date"].tolist())


RECORDS WITH MISSING VALUES


,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
1235,2026-11-04,NaN,NaN,0.0,2493.800049,2420.0,2481.399902,0.0,3140927



RECORDS WITH ZERO VOLUME


,Date,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
832,2025-03-18,3299.900146,3497.699951,0.0,3497.699951,3497.699951,3497.699951,0.0,0
1038,2026-01-15,3060.415527,3192.500000,0.0,3192.500000,3192.500000,3192.500000,0.0,0



DATES WITH MISSING VALUES:
[Timestamp('2026-11-04 00:00:00')]

DATES WITH ZERO VOLUME:
[Timestamp('2025-03-18 00:00:00'), Timestamp('2026-01-15 00:00:00')]


In [5]:

# Define the dates we want to investigate
problem_dates = [
    "2026-11-04",
    "2025-03-18",
    "2026-01-15"
]

# Select the columns relevant to our investigation
columns_to_inspect = [
    "Date", "Open", "High", "Low",
    "Close", "Adj Close", "Volume"
]

# Inspect each problematic date individually
for date_string in problem_dates:

    # Convert the date string into a Pandas timestamp
    target_date = pd.Timestamp(date_string)

    # Find the row position corresponding to that date
    matching_positions = tcs_data.index[
        tcs_data["Date"] == target_date
    ].tolist()

    # Display the date being investigated
    print(f"\nInvestigating: {date_string}")

    # Check whether the date exists in our dataset
    if not matching_positions:
        print("Date not found in the dataset.")
        continue

    # Retrieve the position of the problematic row
    position = matching_positions[0]

    # Select two observations before the problematic date
    start_position = max(position - 2, 0)

    # Select two observations after the problematic date
    end_position = min(position + 3, len(tcs_data))

    # Extract the surrounding trading records
    surrounding_data = tcs_data.iloc[
        start_position:end_position
    ][columns_to_inspect]

    # Display the records in chronological order
    display(surrounding_data.reset_index(drop=True))



Investigating: 2026-11-04


,Date,Open,High,Low,Close,Adj Close,Volume
0,2026-11-02,2523.000000,2549.899902,2492.899902,2501.699951,2501.699951,2817192
1,2026-11-03,2520.000000,2532.000000,2480.000000,2499.100098,2499.100098,3488517
2,2026-11-04,2481.399902,2493.800049,2420.000000,NaN,NaN,3140927



Investigating: 2025-03-18


,Date,Open,High,Low,Close,Adj Close,Volume
0,2025-03-13,3502.300049,3524.300049,3484.100098,3511.050049,3312.495850,1693004
1,2025-03-17,3489.000000,3530.449951,3477.350098,3497.699951,3299.900146,2690906
2,2025-03-18,3497.699951,3497.699951,3497.699951,3497.699951,3299.900146,0
3,2025-03-19,3547.000000,3547.000000,3462.750000,3497.100098,3299.334717,3051244
4,2025-03-20,3512.000000,3578.949951,3512.000000,3563.149902,3361.649170,2845192



Investigating: 2026-01-15


,Date,Open,High,Low,Close,Adj Close,Volume
0,2026-01-13,3245.000000,3280.000000,3210.500000,3268.000000,3132.792236,5645341
1,2026-01-14,3260.000000,3260.000000,3183.000000,3192.500000,3060.415527,3539511
2,2026-01-15,3192.500000,3192.500000,3192.500000,3192.500000,3060.415527,0
3,2026-01-16,3219.399902,3221.899902,3171.000000,3206.699951,3129.910645,4293470
4,2026-01-19,3188.000000,3220.000000,3152.100098,3163.600098,3087.843018,3387562


In [6]:

# Import yfinance to retrieve recent stock prices
import yfinance as yf

# Download the latest 10 days of TCS historical data
recent_tcs = yf.download(
    "TCS.NS",
    period="10d",
    interval="1d",
    auto_adjust=False,
    actions=True,
    progress=False,
    threads=False,
    multi_level_index=False
)

# Check whether Yahoo Finance returned data
if recent_tcs.empty:
    print("No recent TCS data received. Check the connection.")
else:

    # Name the date index
    recent_tcs.index.name = "Date"

    # Display the latest available trading records
    print("RECENT TCS DATA")
    display(recent_tcs.tail(10))

    # Check for missing closing prices
    print("\nMISSING CLOSING PRICES")
    display(
        recent_tcs[recent_tcs["Close"].isna()]
    )

    # Display the most recent available date
    print("\nLatest available date:", recent_tcs.index.max())


RECENT TCS DATA


c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\multi.py:220: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.rename(columns=ctx.isins, inplace=True)
c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\multi.py:224: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs

,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
Date,,,,,,,,
2026-10-22,2506.899902,2506.899902,0.0,2533.000000,2479.000000,2520.000000,0.0,4174671
2026-10-23,2530.000000,2530.000000,0.0,2625.000000,2517.199951,2568.000000,0.0,6857942
2026-10-26,2535.000000,2535.000000,0.0,2549.699951,2491.600098,2500.000000,0.0,4326029
2026-10-27,2496.899902,2496.899902,0.0,2534.899902,2465.399902,2514.100098,0.0,4337081
2026-10-28,2460.000000,2460.000000,0.0,2461.300049,2408.199951,2449.000000,0.0,2540047
2026-10-29,2478.000000,2478.000000,0.0,2509.500000,2464.000000,2490.000000,0.0,3427569
2026-10-30,2502.600098,2502.600098,0.0,2527.699951,2494.199951,2495.199951,0.0,3016983
2026-11-02,2501.699951,2501.699951,0.0,2549.899902,2492.899902,2523.000000,0.0,2817192
2026-11-03,2499.100098,2499.100098,0.0,2532.000000,2480.000000,2520.000000,0.0,3488517



MISSING CLOSING PRICES


,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
Date,,,,,,,,
2026-11-04,NaN,NaN,0.0,2493.800049,2420.0,2481.399902,0.0,3140927



Latest available date: 2026-11-04 00:00:00


In [7]:

# Download recent TCS data with Yahoo Finance price repair enabled
repaired_tcs = yf.download(
    tickers="TCS.NS",
    period="10d",
    interval="1d",
    auto_adjust=False,
    actions=True,
    progress=False,
    threads=False,
    multi_level_index=False,
    repair=True,
    keepna=True,
    timeout=30
)

# Define the date we want to investigate
target_date = "2026-11-04"

# Display the downloaded data after attempting repair
print("RECENT TCS DATA AFTER PRICE REPAIR")
display(repaired_tcs.tail(10))

# Find the specific November 4 record
repair_row = repaired_tcs.loc[
    repaired_tcs.index.strftime("%Y-%m-%d") == target_date
]

# Display the record
print("\nREPAIRED RECORD FOR:", target_date)
display(repair_row)

# Check whether the closing price is still missing
if repair_row.empty:
    print("Target date was not returned by Yahoo Finance.")
else:
    close_value = repair_row["Close"].iloc[0]

    if pd.isna(close_value):
        print("Closing price is STILL missing after repair.")
    else:
        print("Closing price successfully retrieved:", close_value)

# Check whether yfinance marked any record as repaired
if "Repaired?" in repaired_tcs.columns:
    print("\nREPAIR STATUS")
    display(repaired_tcs["Repaired?"].tail(10))


c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/what

RECENT TCS DATA AFTER PRICE REPAIR


c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\multi.py:220: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.rename(columns=ctx.isins, inplace=True)
c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\multi.py:224: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.sort_index(level=0, axis=1, inplace=True)


,Adj Close,Close,Dividends,High,Low,Open,Repaired?,Stock Splits,Volume
Date,,,,,,,,,
2026-10-22,2506.899902,2506.899902,0.0,2533.000000,2479.000000,2520.000000,False,0.0,4174671
2026-10-23,2530.000000,2530.000000,0.0,2625.000000,2517.199951,2568.000000,False,0.0,6857942
2026-10-26,2535.000000,2535.000000,0.0,2549.699951,2491.600098,2500.000000,False,0.0,4326029
2026-10-27,2496.899902,2496.899902,0.0,2534.899902,2465.399902,2514.100098,False,0.0,4337081
2026-10-28,2460.000000,2460.000000,0.0,2461.300049,2408.199951,2449.000000,False,0.0,2540047
2026-10-29,2478.000000,2478.000000,0.0,2509.500000,2464.000000,2490.000000,False,0.0,3427569
2026-10-30,2502.600098,2502.600098,0.0,2527.699951,2494.199951,2495.199951,False,0.0,3016983
2026-11-02,2501.699951,2501.699951,0.0,2549.899902,2492.899902,2523.000000,False,0.0,2817192
2026-11-03,2499.100098,2499.100098,0.0,2532.000000,2480.000000,2520.000000,False,0.0,3488517



REPAIRED RECORD FOR: 2026-11-04


,Adj Close,Close,Dividends,High,Low,Open,Repaired?,Stock Splits,Volume
Date,,,,,,,,,
2026-11-04,2436.0,2436.0,0.0,2493.800049,2420.0,2481.399902,True,0.0,3140927


Closing price successfully retrieved: 2436.0

REPAIR STATUS


Date
2026-10-22    False
2026-10-23    False
2026-10-26    False
2026-10-27    False
2026-10-28    False
2026-10-29    False
2026-10-30    False
2026-11-02    False
2026-11-03    False
2026-11-04     True
Name: Repaired?, dtype: bool

In [8]:
# Download five years of TCS historical data with price repair enabled
full_tcs_repaired = yf.download(
    tickers="TCS.NS",
    period="5y",
    interval="1d",
    auto_adjust=False,
    actions=True,
    repair=True,
    keepna=True,
    progress=False,
    threads=False,
    multi_level_index=False
)

# Check whether data was received
if full_tcs_repaired.empty:
    print("Download failed. No data received.")
else:
    # Display the total number of downloaded records
    print("Total records:", len(full_tcs_repaired))

    # Display the repaired record for November 4
    print("\nTCS DATA FOR 4 NOVEMBER 2026:")

    display(
        full_tcs_repaired.loc["2026-11-04":"2026-11-04"]
    )

    # Count remaining missing closing prices
    print(
        "\nMissing closing prices:",
        full_tcs_repaired["Close"].isna().sum()
    )

c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\utils.py:585: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dividends.set_index("date", inplace=True)
c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\utils.py:587: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs

Total records: 1236

TCS DATA FOR 4 NOVEMBER 2026:


c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\multi.py:220: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.rename(columns=ctx.isins, inplace=True)
c:\Users\HP\Documents\githubprojects\stock-market-analytics\.venv\Lib\site-packages\yfinance\multi.py:224: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.sort_index(level=0, axis=1, inplace=True)


,Adj Close,Close,Dividends,High,Low,Open,Repaired?,Stock Splits,Volume
Date,,,,,,,,,
2026-11-04,2436.0,2436.0,0.0,2493.800049,2420.0,2481.399902,True,0.0,3140927



Missing closing prices: 0


In [10]:

# Import tools for locating folders and Python modules
from pathlib import Path
import sys

# Find the current notebook working directory
current_dir = Path.cwd().resolve()

# Search the current directory and its parent folders
# to locate the main project directory
PROJECT_ROOT = next(
    (
        folder
        for folder in [current_dir, *current_dir.parents]
        if (folder / "src" / "config.py").exists()
    ),
    None
)

# Stop if the project root cannot be identified
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

# Add the project root to Python's module search path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import the processed data folder from our configuration
from src.config import PROCESSED_DATA_DIR

# Create the processed folder if it does not exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Define the output CSV file
repaired_file_path = PROCESSED_DATA_DIR / "TCS_NS_repaired.csv"

# Name the date index
full_tcs_repaired.index.name = "Date"

# Save the complete five-year repaired dataset
full_tcs_repaired.to_csv(repaired_file_path, index=True)

# Display the results
print("Repaired TCS dataset saved successfully!")
print("File location:", repaired_file_path)
print("File exists:", repaired_file_path.exists())
print("Total records saved:", len(full_tcs_repaired))


Repaired TCS dataset saved successfully!
File location: C:\Users\HP\Documents\githubprojects\stock-market-analytics\data\processed\TCS_NS_repaired.csv
File exists: True
Total records saved: 1236


In [11]:
# Import libraries for reading the saved CSV
import pandas as pd
from pathlib import Path

# Locate the main project folder
project_root = next(
    folder
    for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (folder / "src" / "config.py").exists()
)

# Define the saved CSV location
saved_path = project_root / "data" / "processed" / "TCS_NS_repaired.csv"

# Read the saved CSV into a DataFrame
saved_tcs = pd.read_csv(saved_path, parse_dates=["Date"])

# Display the number of saved records
print("Saved records:", len(saved_tcs))

# Display the number of missing closing prices
print("Missing closing prices:", saved_tcs["Close"].isna().sum())

# Inspect November 4, 2026
target_record = saved_tcs[
    saved_tcs["Date"] == pd.Timestamp("2026-11-04")
]

print("\nNovember 4, 2026 record:")
display(target_record)

# Check whether the record was marked as repaired
if "Repaired?" in saved_tcs.columns and not target_record.empty:
    print("Repair status:", target_record["Repaired?"].iloc[0])

Saved records: 1236
Missing closing prices: 0

November 4, 2026 record:


,Date,Adj Close,Close,Dividends,High,Low,Open,Repaired?,Stock Splits,Volume
1235,2026-11-04,2436.0,2436.0,0.0,2493.800049,2420.0,2481.399902,True,0.0,3140927


Repair status: True


In [12]:

# Import Path to locate our project folders
from pathlib import Path

# Find the main project directory
project_root = next(
    folder
    for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (folder / "src" / "config.py").exists()
)

# Define the original and repaired CSV locations
raw_path = project_root / "data" / "raw" / "TCS_NS.csv"
repaired_path = project_root / "data" / "processed" / "TCS_NS_repaired.csv"

# Load both datasets
raw_tcs = pd.read_csv(raw_path, parse_dates=["Date"])
saved_repaired_tcs = pd.read_csv(repaired_path, parse_dates=["Date"])

# Compare the record counts
print("ORIGINAL RECORDS:", len(raw_tcs))
print("REPAIRED RECORDS:", len(saved_repaired_tcs))

# Compare the date ranges
print("\nORIGINAL DATE RANGE:")
print(raw_tcs["Date"].min(), "to", raw_tcs["Date"].max())

print("\nREPAIRED DATE RANGE:")
print(saved_repaired_tcs["Date"].min(), "to", saved_repaired_tcs["Date"].max())

# Identify dates present in one dataset but not the other
raw_dates = set(raw_tcs["Date"])
repaired_dates = set(saved_repaired_tcs["Date"])

print("\nDATES ONLY IN ORIGINAL:", len(raw_dates - repaired_dates))
print("DATES ONLY IN REPAIRED:", len(repaired_dates - raw_dates))

# Compare missing closing prices
print("\nORIGINAL MISSING CLOSE:", raw_tcs["Close"].isna().sum())
print("REPAIRED MISSING CLOSE:", saved_repaired_tcs["Close"].isna().sum())

# Check duplicate dates in both datasets
print("\nORIGINAL DUPLICATE DATES:", raw_tcs["Date"].duplicated().sum())
print("REPAIRED DUPLICATE DATES:", saved_repaired_tcs["Date"].duplicated().sum())


ORIGINAL RECORDS: 1236
REPAIRED RECORDS: 1236

ORIGINAL DATE RANGE:
2021-11-04 00:00:00 to 2026-11-04 00:00:00

REPAIRED DATE RANGE:
2021-11-04 00:00:00 to 2026-11-04 00:00:00

DATES ONLY IN ORIGINAL: 0
DATES ONLY IN REPAIRED: 0

ORIGINAL MISSING CLOSE: 1
REPAIRED MISSING CLOSE: 0

ORIGINAL DUPLICATE DATES: 0
REPAIRED DUPLICATE DATES: 0


In [13]:

# Convert trading volume into numeric values
volume_values = pd.to_numeric(
    saved_repaired_tcs["Volume"],
    errors="coerce"
)

# Find all records with zero trading volume
zero_volume_positions = [
    position
    for position, is_zero in enumerate(volume_values.eq(0))
    if is_zero
]

# Select the important columns for inspection
columns_to_show = [
    "Date", "Open", "High", "Low",
    "Close", "Adj Close", "Volume"
]

# Include repair status if available
if "Repaired?" in saved_repaired_tcs.columns:
    columns_to_show.append("Repaired?")

# Display the total number of zero-volume records
print("Zero-volume records:", len(zero_volume_positions))

# Inspect each zero-volume record and its neighbouring dates
for position in zero_volume_positions:

    # Select two records before and after the zero-volume date
    start = max(position - 2, 0)
    end = min(position + 3, len(saved_repaired_tcs))

    # Extract the surrounding records
    surrounding_records = saved_repaired_tcs.iloc[
        start:end
    ][columns_to_show]

    # Display the surrounding trading history
    print("\nInvestigating zero-volume record:")
    display(surrounding_records.reset_index(drop=True))


Zero-volume records: 2

Investigating zero-volume record:


,Date,Open,High,Low,Close,Adj Close,Volume,Repaired?
0,2025-03-13,3502.300049,3524.300049,3484.100098,3511.050049,3312.495850,1693004,False
1,2025-03-17,3489.000000,3530.449951,3477.350098,3497.699951,3299.900635,2690906,False
2,2025-03-18,3497.699951,3497.699951,3497.699951,3497.699951,3299.900635,0,False
3,2025-03-19,3547.000000,3547.000000,3462.750000,3497.100098,3299.334717,3051244,False
4,2025-03-20,3512.000000,3578.949951,3512.000000,3563.149902,3361.649414,2845192,False



Investigating zero-volume record:


,Date,Open,High,Low,Close,Adj Close,Volume,Repaired?
0,2026-01-13,3245.000000,3280.000000,3210.500000,3268.000000,3132.791992,5645341,False
1,2026-01-14,3260.000000,3260.000000,3183.000000,3192.500000,3060.415771,3539511,False
2,2026-01-15,3192.500000,3192.500000,3192.500000,3192.500000,3060.415771,0,False
3,2026-01-16,3219.399902,3221.899902,3171.000000,3206.699951,3129.910645,4293470,False
4,2026-01-19,3188.000000,3220.000000,3152.100098,3163.600098,3087.843018,3387562,False


In [14]:

# Set Date as the index for easier date-based lookup
price_data = saved_repaired_tcs.set_index("Date").sort_index()

# Retrieve the record before March 18
previous_day = price_data.loc[pd.Timestamp("2025-03-17")]

# Retrieve the record after March 18
next_day = price_data.loc[pd.Timestamp("2025-03-19")]

# Calculate the adjusted-price factor for March 17
previous_factor = (
    previous_day["Adj Close"] / previous_day["Close"]
)

# Calculate the adjusted-price factor for March 19
next_factor = (
    next_day["Adj Close"] / next_day["Close"]
)

# Display both adjustment factors
print("Adjustment factor on March 17:", previous_factor)
print("Adjustment factor on March 19:", next_factor)

# Check whether both factors are approximately equal
factors_match = abs(previous_factor - next_factor) < 0.000001

print("\nAdjustment factors consistent:", factors_match)

# Calculate an adjusted close only if the factors match
if factors_match:

    # Use the average of the two neighbouring adjustment factors
    adjustment_factor = (previous_factor + next_factor) / 2

    # Apply the factor to the independently verified closing price
    estimated_adjusted_close = 3552.45 * adjustment_factor

    print("Estimated adjusted close:", estimated_adjusted_close)

else:
    print("Adjustment factors differ. Further investigation is required.")


Adjustment factor on March 17: 0.9434487465570113
Adjustment factor on March 19: 0.9434487502968768

Adjustment factors consistent: True
Estimated adjusted close: 3351.5545063492973


In [15]:

# Import Path for saving the cleaned dataset
from pathlib import Path

# Start with a copy of the verified repaired dataset
clean_tcs = saved_repaired_tcs.copy()

# Ensure the Date column uses datetime format
clean_tcs["Date"] = pd.to_datetime(clean_tcs["Date"])

# Define a helper to retrieve exactly one record by date
def get_record(date_string):
    # Find rows matching the requested date
    rows = clean_tcs.loc[
        clean_tcs["Date"] == pd.Timestamp(date_string)
    ]

    # Stop if the date is missing or duplicated
    if len(rows) != 1:
        raise ValueError(f"Expected one record for {date_string}")

    # Return the matching record
    return rows.iloc[0]


# ------------------------------------------------------------
# 1. CORRECT THE MARCH 18, 2025 RECORD
# ------------------------------------------------------------

# Retrieve neighbouring trading records
previous = get_record("2025-03-17")
following = get_record("2025-03-19")

# Calculate the adjustment factor for March 17
factor_before = previous["Adj Close"] / previous["Close"]

# Calculate the adjustment factor for March 19
factor_after = following["Adj Close"] / following["Close"]

# Ensure neighbouring adjustment factors are consistent
if abs(factor_before - factor_after) > 0.000001:
    raise ValueError("Adjustment factors differ. Stop correction.")

# Calculate the adjustment factor for March 18
march_factor = (factor_before + factor_after) / 2

# Identify the March 18 record
march_mask = clean_tcs["Date"] == pd.Timestamp("2025-03-18")

# Preserve its original values for the audit log
old_march = clean_tcs.loc[march_mask].iloc[0].copy()

# Enter independently verified historical OHLCV values
corrected_values = {
    "Open": 3502.35,
    "High": 3556.95,
    "Low": 3478.45,
    "Close": 3552.45,
    "Volume": 3069109
}

# Apply the corrected values to our working dataset
for column, value in corrected_values.items():
    clean_tcs.loc[march_mask, column] = value

# Reconstruct adjusted closing price using the consistent factor
corrected_adj_close = 3552.45 * march_factor

# Update the adjusted closing price
clean_tcs.loc[march_mask, "Adj Close"] = corrected_adj_close


# ------------------------------------------------------------
# 2. EXCLUDE THE JANUARY 15, 2026 MARKET HOLIDAY
# ------------------------------------------------------------

# Identify the January 15 record
holiday_mask = clean_tcs["Date"] == pd.Timestamp("2026-01-15")

# Preserve its original values for the audit log
if holiday_mask.sum() != 1:
    raise ValueError("Expected exactly one January 15 record.")

old_holiday = clean_tcs.loc[holiday_mask].iloc[0].copy()

# Exclude this non-trading date from the analytical dataset
clean_tcs = clean_tcs.loc[~holiday_mask].copy()


# ------------------------------------------------------------
# 3. CREATE THE CORRECTION AUDIT LOG
# ------------------------------------------------------------

# Record the manually corrected March 18 observation
audit_entries = [
    {
        "Date": "2025-03-18",
        "Action": "Externally corrected",
        "Reason": "Yahoo Finance returned a stale zero-volume record.",
        "Source": "Economic Times and EquityPandit",
        "Source_URL": "https://www.equitypandit.com/historical-data/tcs",
        "Old_Close": old_march["Close"],
        "New_Close": 3552.45,
        "Old_Volume": old_march["Volume"],
        "New_Volume": 3069109,
        "Adjusted_Close_Method": "Reconstructed using matching adjacent-day factors"
    },

    # Record the excluded January 15 market holiday
    {
        "Date": "2026-01-15",
        "Action": "Excluded",
        "Reason": "Official NSE equity trading holiday.",
        "Source": "NSE 2026 Equity Holiday Calendar",
        "Source_URL": "https://www.nseindia.com/resources/exchange-communication-holidays?apiversion=v2",
        "Old_Close": old_holiday["Close"],
        "New_Close": None,
        "Old_Volume": old_holiday["Volume"],
        "New_Volume": None,
        "Adjusted_Close_Method": "Not applicable"
    }
]

# Convert audit entries into a Pandas DataFrame
correction_log = pd.DataFrame(audit_entries)


# ------------------------------------------------------------
# 4. SAVE THE CLEANED DATASET AND AUDIT LOG
# ------------------------------------------------------------

# Locate the main project directory
project_root = next(
    folder
    for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (folder / "src" / "config.py").exists()
)

# Define the processed data folder
processed_dir = project_root / "data" / "processed"

# Create the folder if it does not exist
processed_dir.mkdir(parents=True, exist_ok=True)

# Define the cleaned dataset destination
clean_file = processed_dir / "TCS_NS_clean.csv"

# Define the correction log destination
audit_file = processed_dir / "TCS_NS_correction_log.csv"

# Save the cleaned dataset
clean_tcs.to_csv(clean_file, index=False)

# Save the audit log separately
correction_log.to_csv(audit_file, index=False)


# ------------------------------------------------------------
# 5. DISPLAY FINAL RESULTS
# ------------------------------------------------------------

# Convert volume into numeric format
clean_volume = pd.to_numeric(clean_tcs["Volume"], errors="coerce")

# Display the number of remaining records
print("CLEANING COMPLETED")
print("Records before:", len(saved_repaired_tcs))
print("Records after:", len(clean_tcs))

# Display important data quality results
print("Missing closing prices:", clean_tcs["Close"].isna().sum())
print("Duplicate dates:", clean_tcs["Date"].duplicated().sum())
print("Zero-volume records:", (clean_volume == 0).sum())

# Display corrected March record
print("\nCORRECTED MARCH 18 RECORD")
display(clean_tcs.loc[march_mask])

# Display saved file locations
print("\nCLEANED FILE:", clean_file)
print("AUDIT LOG:", audit_file)
print("Cleaned file exists:", clean_file.exists())
print("Audit file exists:", audit_file.exists())


CLEANING COMPLETED
Records before: 1236
Records after: 1235
Missing closing prices: 0
Duplicate dates: 0
Zero-volume records: 0

CORRECTED MARCH 18 RECORD


,Date,Adj Close,Close,Dividends,High,Low,Open,Repaired?,Stock Splits,Volume
832,2025-03-18,3351.554506,3552.45,0.0,3556.95,3478.45,3502.35,False,0.0,3069109



CLEANED FILE: C:\Users\HP\Documents\githubprojects\stock-market-analytics\data\processed\TCS_NS_clean.csv
AUDIT LOG: C:\Users\HP\Documents\githubprojects\stock-market-analytics\data\processed\TCS_NS_correction_log.csv
Cleaned file exists: True
Audit file exists: True


In [16]:
# Load the cleaned CSV from disk
final_tcs = pd.read_csv(clean_file, parse_dates=["Date"])

# Load the correction audit log
final_audit = pd.read_csv(audit_file)

# Convert volume into numeric values
final_volume = pd.to_numeric(final_tcs["Volume"], errors="coerce")

# Display the final number of records
print("FINAL TCS DATA QUALITY REPORT")

print("Total records:", len(final_tcs))

# Check for missing closing prices
print("Missing closing prices:", final_tcs["Close"].isna().sum())

# Check for duplicate dates
print("Duplicate dates:", final_tcs["Date"].duplicated().sum())

# Check for zero-volume records
print("Zero-volume records:", (final_volume == 0).sum())

# Check for invalid prices
price_columns = ["Open", "High", "Low", "Close", "Adj Close"]
print("Missing price values:", final_tcs[price_columns].isna().sum().sum())

# Display the correction audit log
print("\nCORRECTION AUDIT LOG")
display(final_audit)

# Confirm the saved file locations
print("\nClean CSV exists:", clean_file.exists())
print("Audit CSV exists:", audit_file.exists())

FINAL TCS DATA QUALITY REPORT
Total records: 1235
Missing closing prices: 0
Duplicate dates: 0
Zero-volume records: 0
Missing price values: 0

CORRECTION AUDIT LOG


,Date,Action,Reason,Source,Source_URL,Old_Close,New_Close,Old_Volume,New_Volume,Adjusted_Close_Method
0,2025-03-18,Externally corrected,Yahoo Finance returned a stale zero-volume rec...,Economic Times and EquityPandit,https://www.equitypandit.com/historical-data/tcs,3497.699951,3552.45,0,3069109.0,Reconstructed using matching adjacent-day factors
1,2026-01-15,Excluded,Official NSE equity trading holiday.,NSE 2026 Equity Holiday Calendar,https://www.nseindia.com/resources/exchange-co...,3192.500000,NaN,0,NaN,Not applicable



Clean CSV exists: True
Audit CSV exists: True


In [17]:
# Reload the cleaned dataset from disk
final_tcs = pd.read_csv(clean_file, parse_dates=["Date"])

# Convert volume to numeric
volume = pd.to_numeric(final_tcs["Volume"], errors="coerce")

# Display final quality metrics
print("FINAL DATA QUALITY REPORT")
print("Total records:", len(final_tcs))
print("Missing closing prices:", final_tcs["Close"].isna().sum())
print("Duplicate dates:", final_tcs["Date"].duplicated().sum())
print("Zero-volume records:", (volume == 0).sum())

# Check whether the excluded holiday is absent
print(
    "January 15 excluded:",
    not (final_tcs["Date"] == "2026-01-15").any()
)

# Verify the March correction
print("\nCORRECTED MARCH RECORD")

display(
    final_tcs[
        final_tcs["Date"] == "2025-03-18"
    ][["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]]
)

# Count entries in the audit log
print("\nAudit entries:", len(pd.read_csv(audit_file)))

FINAL DATA QUALITY REPORT
Total records: 1235
Missing closing prices: 0
Duplicate dates: 0
Zero-volume records: 0
January 15 excluded: True

CORRECTED MARCH RECORD


,Date,Open,High,Low,Close,Adj Close,Volume
832,2025-03-18,3502.35,3556.95,3478.45,3552.45,3351.554506,3069109



Audit entries: 2


In [18]:

# Import pandas for reading the saved stock datasets
import pandas as pd

# Import the reusable cleaning functions
from src.data_cleaning import clean_stock_data, save_cleaning_outputs

# Import the processed data directory
from src.config import PROCESSED_DATA_DIR


# ------------------------------------------------------------
# 1. LOAD THE REPAIRED TCS DATASET
# ------------------------------------------------------------

# Define the location of the repaired TCS CSV
repaired_path = PROCESSED_DATA_DIR / "TCS_NS_repaired.csv"

# Read the repaired dataset into a DataFrame
test_tcs = pd.read_csv(repaired_path, parse_dates=["Date"])

# Display the number of input records
print("Input TCS records:", len(test_tcs))


# ------------------------------------------------------------
# 2. CALCULATE THE ADJUSTED PRICE FACTOR
# ------------------------------------------------------------

# Retrieve the closing price and adjusted closing price
# from the previous trading session
previous = test_tcs.loc[
    test_tcs["Date"] == pd.Timestamp("2025-03-17")
].iloc[0]

# Retrieve the corresponding values from the next session
following = test_tcs.loc[
    test_tcs["Date"] == pd.Timestamp("2025-03-19")
].iloc[0]

# Calculate the adjustment factor before the correction date
factor_before = previous["Adj Close"] / previous["Close"]

# Calculate the adjustment factor after the correction date
factor_after = following["Adj Close"] / following["Close"]

# Verify that the factors are sufficiently close
if abs(factor_before - factor_after) > 0.000001:
    raise ValueError("Adjustment factors are inconsistent.")

# Reconstruct the adjusted closing price
adjusted_close = 3552.45 * (
    (factor_before + factor_after) / 2
)


# ------------------------------------------------------------
# 3. DEFINE THE VERIFIED CORRECTION
# ------------------------------------------------------------

# Supply the corrected March 18 OHLCV and adjusted close values
corrections = {
    "2025-03-18": {
        "values": {
            "Open": 3502.35,
            "High": 3556.95,
            "Low": 3478.45,
            "Close": 3552.45,
            "Adj Close": adjusted_close,
            "Volume": 3069109
        },

        # Explain why the correction is required
        "reason": "Externally verified correction of stale zero-volume data.",

        # Identify the historical sources
        "source": "Economic Times and EquityPandit",

        # Record the source URL
        "source_url": "https://www.equitypandit.com/historical-data/tcs"
    }
}


# ------------------------------------------------------------
# 4. DEFINE THE VERIFIED MARKET HOLIDAY
# ------------------------------------------------------------

# Supply the NSE holiday that must be excluded
exclusions = {
    "2026-01-15": {
        "reason": "Official NSE equity trading holiday.",

        # Record the source used to verify the holiday
        "source": "NSE 2026 Equity Holiday Calendar",

        # Store the official NSE holiday resource
        "source_url": "https://www.nseindia.com/resources/exchange-communication-holidays?apiversion=v2"
    }
}


# ------------------------------------------------------------
# 5. RUN OUR REUSABLE CLEANING FUNCTION
# ------------------------------------------------------------

# Call the reusable function with the repaired TCS dataset
# and the independently verified correction and exclusion rules
module_cleaned, module_audit, module_report = clean_stock_data(
    data=test_tcs,
    ticker="TCS.NS",
    corrections=corrections,
    exclusions=exclusions
)

# Display the final data-quality report
print("\nREUSABLE CLEANING MODULE RESULTS")

# Show all quality metrics
for metric, value in module_report.items():
    print(f"{metric}: {value}")

# Display the correction and exclusion audit records
print("\nMODULE AUDIT LOG")
display(module_audit)


# ------------------------------------------------------------
# 6. SAVE THE TEST OUTPUT SEPARATELY
# ------------------------------------------------------------

# Create a separate folder for testing the reusable module
test_output_dir = PROCESSED_DATA_DIR / "module_test"

# Save the cleaned data and audit log without overwriting
# our previously created manual TCS cleaning outputs
module_clean_path, module_audit_path = save_cleaning_outputs(
    cleaned_data=module_cleaned,
    audit_log=module_audit,
    ticker="TCS.NS",
    output_dir=test_output_dir
)

# Confirm that the files were successfully created
print("\nMODULE TEST FILES")
print("Cleaned CSV exists:", module_clean_path.exists())
print("Audit CSV exists:", module_audit_path.exists())


# ------------------------------------------------------------
# 7. COMPARE AUTOMATED AND MANUAL CLEANING RESULTS
# ------------------------------------------------------------

# Load the previously generated manual cleaning dataset
manual_clean = pd.read_csv(
    PROCESSED_DATA_DIR / "TCS_NS_clean.csv",
    parse_dates=["Date"]
)

# Compare the final record counts
print("\nCOMPARISON WITH MANUAL CLEANING")
print("Manual records:", len(manual_clean))
print("Automated records:", len(module_cleaned))

# Verify that both datasets contain the same trading dates
print(
    "Trading dates match:",
    manual_clean["Date"].equals(module_cleaned["Date"])
)

# Compare the closing prices
print(
    "Closing prices match:",
    manual_clean["Close"].equals(module_cleaned["Close"])
)

# Compare the adjusted closing prices
print(
    "Adjusted closing prices match:",
    manual_clean["Adj Close"].equals(module_cleaned["Adj Close"])
)


Input TCS records: 1236

REUSABLE CLEANING MODULE RESULTS
Ticker: TCS.NS
Total Records: 1235
Missing Closing Prices: 0
Missing Price Values: 0
Duplicate Dates: 0
Zero Volume Records: 0
Negative Volume Records: 0
Invalid OHLC Records: 0
Chronological Order: True

MODULE AUDIT LOG


,Ticker,Date,Action,Reason,Source,Source URL,Old Values,New Values
0,TCS.NS,2025-03-18,Corrected,Externally verified correction of stale zero-v...,Economic Times and EquityPandit,https://www.equitypandit.com/historical-data/tcs,"{""Open"": 3497.699951171875, ""High"": 3497.69995...","{""Open"": 3502.35, ""High"": 3556.95, ""Low"": 3478..."
1,TCS.NS,2026-01-15,Excluded,Official NSE equity trading holiday.,NSE 2026 Equity Holiday Calendar,https://www.nseindia.com/resources/exchange-co...,"{""Date"": ""2026-01-15 00:00:00"", ""Open"": 3192.5...",{}



MODULE TEST FILES
Cleaned CSV exists: True
Audit CSV exists: True

COMPARISON WITH MANUAL CLEANING
Manual records: 1235
Automated records: 1235
Trading dates match: True
Closing prices match: True
Adjusted closing prices match: True


In [19]:

# Import libraries for inspecting suspicious stock records
import pandas as pd
from pathlib import Path

# Locate the main project directory
project_root = next(
    folder
    for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (folder / "src" / "config.py").exists()
)

# Locate the folder containing stocks that require review
review_dir = (
    project_root / "data" / "processed" /
    "batch" / "review_required"
)

# Find all cleaned stock CSV files requiring review
review_files = sorted(review_dir.glob("*_clean.csv"))

# Create an empty list to collect suspicious observations
suspicious_records = []

# Inspect each stock dataset individually
for file_path in review_files:

    # Read the cleaned historical stock dataset
    stock_data = pd.read_csv(file_path, parse_dates=["Date"])

    # Convert trading volume into numeric values
    stock_data["Volume"] = pd.to_numeric(
        stock_data["Volume"],
        errors="coerce"
    )

    # Find positions where trading volume is zero
    zero_positions = stock_data.index[
        stock_data["Volume"] == 0
    ].tolist()

    # Inspect each zero-volume observation
    for position in zero_positions:

        # Retrieve the suspicious trading record
        current = stock_data.loc[position]

        # Retrieve the previous available trading session
        previous = (
            stock_data.loc[position - 1]
            if position > 0 else None
        )

        # Retrieve the next available trading session
        following = (
            stock_data.loc[position + 1]
            if position < len(stock_data) - 1 else None
        )

        # Store the important financial information
        suspicious_records.append({
            "Ticker": file_path.stem.replace("_clean", ""),
            "Date": current["Date"],
            "Open": current["Open"],
            "High": current["High"],
            "Low": current["Low"],
            "Close": current["Close"],
            "Adj Close": current["Adj Close"],
            "Volume": current["Volume"],
            "Previous Date": previous["Date"] if previous is not None else None,
            "Previous Close": previous["Close"] if previous is not None else None,
            "Next Date": following["Date"] if following is not None else None,
            "Next Close": following["Close"] if following is not None else None
        })

# Convert the collected observations into a DataFrame
zero_volume_report = pd.DataFrame(suspicious_records)

# Display the investigation results
print("STOCKS REQUIRING ZERO-VOLUME INVESTIGATION")
print("Review files:", len(review_files))
print("Zero-volume observations:", len(zero_volume_report))

# Show every suspicious record and its neighbouring prices
if not zero_volume_report.empty:
    display(zero_volume_report.sort_values(["Ticker", "Date"]))
else:
    print("No zero-volume observations found.")


STOCKS REQUIRING ZERO-VOLUME INVESTIGATION
Review files: 18
Zero-volume observations: 19


,Ticker,Date,Open,High,Low,Close,Adj Close,Volume,Previous Date,Previous Close,Next Date,Next Close
0,BAJAJ_AUTO_NS,2024-01-15,7301.950195,7301.950195,7301.950195,7301.950195,6966.017578,0,2024-01-12,7301.950195,2024-01-16,7293.950195
1,BAJAJ_AUTO_NS,2025-03-18,7486.049805,7486.049805,7486.049805,7486.049805,7199.689941,0,2025-03-17,7486.049805,2025-03-19,7721.850098
2,BAJFINANCE_NS,2025-03-18,858.034973,858.034973,858.034973,858.034973,847.562744,0,2025-03-17,858.034973,2025-03-19,873.159973
3,BHARTIARTL_NS,2025-03-18,1639.150024,1639.150024,1639.150024,1639.150024,1605.356689,0,2025-03-17,1639.150024,2025-03-19,1637.150024
4,CIPLA_NS,2025-03-18,1492.599976,1492.599976,1492.599976,1492.599976,1463.097900,0,2025-03-17,1492.599976,2025-03-19,1499.150024
5,HCLTECH_NS,2025-03-18,1545.150024,1545.150024,1545.150024,1545.150024,1437.194214,0,2025-03-17,1545.150024,2025-03-19,1544.449951
6,HDFCBANK_NS,2025-03-18,855.200012,855.200012,855.200012,855.200012,830.065613,0,2025-03-17,855.200012,2025-03-19,872.049988
7,HINDUNILVR_NS,2025-03-18,2171.250000,2171.250000,2171.250000,2171.250000,2110.429199,0,2025-03-17,2171.250000,2025-03-19,2200.050049
8,ICICIBANK_NS,2025-03-18,1269.000000,1269.000000,1269.000000,1269.000000,1248.755737,0,2025-03-17,1269.000000,2025-03-19,1313.099976
9,INFY_NS,2025-03-18,1590.050049,1590.050049,1590.050049,1590.050049,1477.782959,0,2025-03-17,1590.050049,2025-03-19,1586.550049


In [20]:

# Import libraries for downloading and reading the NSE archive
import requests
import zipfile
import pandas as pd

from io import BytesIO
from pathlib import Path

# Locate the main project directory
project_root = next(
    folder
    for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (folder / "src" / "config.py").exists()
)

# Create a separate folder for official reference data
reference_dir = project_root / "data" / "raw" / "reference"
reference_dir.mkdir(parents=True, exist_ok=True)

# Official NSE UDiFF Bhavcopy archive for March 18, 2025
url = (
    "https://nsearchives.nseindia.com/content/cm/"
    "BhavCopy_NSE_CM_0_0_0_20250318_F_0000.csv.zip"
)

# Prepare browser-like headers for the NSE request
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/zip,application/octet-stream,*/*",
    "Referer": "https://www.nseindia.com/"
}

# Create an HTTP session
session = requests.Session()

try:

    # Visit the NSE homepage to establish a session
    session.get(
        "https://www.nseindia.com",
        headers=headers,
        timeout=30
    )

    # Download the official historical archive
    response = session.get(
        url,
        headers=headers,
        timeout=60
    )

    # Stop if the download was unsuccessful
    response.raise_for_status()

    # Check whether the downloaded content is a valid ZIP archive
    archive_data = BytesIO(response.content)

    if not zipfile.is_zipfile(archive_data):
        raise ValueError(
            "The NSE response is not a valid ZIP archive."
        )

    # Save the original archive for future reproducibility
    archive_path = (
        reference_dir / "NSE_Bhavcopy_20250318.zip"
    )

    archive_path.write_bytes(response.content)

    # Read the CSV file inside the downloaded ZIP archive
    with zipfile.ZipFile(archive_data) as archive:

        # Identify CSV files inside the archive
        csv_files = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]

        # Ensure a CSV file is available
        if not csv_files:
            raise ValueError("No CSV found inside the NSE archive.")

        # Load the official market data
        with archive.open(csv_files[0]) as csv_file:
            nse_bhavcopy = pd.read_csv(csv_file)

    # Display download and file information
    print("OFFICIAL NSE BHAVCOPY DOWNLOADED")
    print("Reference date: 18 March 2025")
    print("Archive saved:", archive_path)
    print("Total exchange records:", len(nse_bhavcopy))

    # Display actual column names
    print("\nNSE BHAVCOPY COLUMNS:")
    print(nse_bhavcopy.columns.tolist())

    # Display the first five records
    print("\nFIRST FIVE EXCHANGE RECORDS:")
    display(nse_bhavcopy.head())

except requests.RequestException as error:

    # Display network and download problems
    print("NSE DOWNLOAD ERROR:", error)

except (ValueError, zipfile.BadZipFile) as error:

    # Display archive validation problems
    print("FILE VALIDATION ERROR:", error)


OFFICIAL NSE BHAVCOPY DOWNLOADED
Reference date: 18 March 2025
Archive saved: C:\Users\HP\Documents\githubprojects\stock-market-analytics\data\raw\reference\NSE_Bhavcopy_20250318.zip
Total exchange records: 3043

NSE BHAVCOPY COLUMNS:
['TradDt', 'BizDt', 'Sgmt', 'Src', 'FinInstrmTp', 'FinInstrmId', 'ISIN', 'TckrSymb', 'SctySrs', 'XpryDt', 'FininstrmActlXpryDt', 'StrkPric', 'OptnTp', 'FinInstrmNm', 'OpnPric', 'HghPric', 'LwPric', 'ClsPric', 'LastPric', 'PrvsClsgPric', 'UndrlygPric', 'SttlmPric', 'OpnIntrst', 'ChngInOpnIntrst', 'TtlTradgVol', 'TtlTrfVal', 'TtlNbOfTxsExctd', 'SsnId', 'NewBrdLotQty', 'Rmks', 'Rsvd1', 'Rsvd2', 'Rsvd3', 'Rsvd4']

FIRST FIVE EXCHANGE RECORDS:


,TradDt,BizDt,Sgmt,Src,FinInstrmTp,FinInstrmId,ISIN,TckrSymb,SctySrs,XpryDt,...,TtlTradgVol,TtlTrfVal,TtlNbOfTxsExctd,SsnId,NewBrdLotQty,Rmks,Rsvd1,Rsvd2,Rsvd3,Rsvd4
0,2025-03-18,2025-03-18,CM,NSE,STK,19078,IN0020200104,SGBJUN28,GB,NaN,...,205,1775655.85,32,F1,1,NaN,NaN,NaN,NaN,NaN
1,2025-03-18,2025-03-18,CM,NSE,STK,1342,IN0020200286,SGBN28VIII,GB,NaN,...,277,2403518.61,53,F1,1,NaN,NaN,NaN,NaN,NaN
2,2025-03-18,2025-03-18,CM,NSE,STK,3742,IN0020210053,SGBMAY29I,GB,NaN,...,1946,16826197.19,227,F1,1,NaN,NaN,NaN,NaN,NaN
3,2025-03-18,2025-03-18,CM,NSE,STK,3743,IN0020210061,SGBJUN29II,GB,NaN,...,1590,13726007.99,81,F1,1,NaN,NaN,NaN,NaN,NaN
4,2025-03-18,2025-03-18,CM,NSE,STK,3800,IN0020210087,SGBJU29III,GB,NaN,...,385,3328849.31,54,F1,1,NaN,NaN,NaN,NaN,NaN
